# Warehouse Operations AI Copilot

This prototype uses simulated warehouse data to demonstrate inventory and demand analytics, replenishment recommendations, demand forecasting, FDS retrieval (RAG), Gemini tool calling, conversational memory, historical stockout pattern detection and a dashboard.

**Important scope:** this is a decision-support prototype, not a live WMS integration. It does not execute warehouse transactions. Operational recommendations require human review and approval.

## Before running
1. Run the notebook from top to bottom in a fresh Google Colab runtime.
2. Upload the Excel training workbook when prompted.
3. Add `GEMINI_API_KEY` to Colab Secrets (do not paste the key into notebook code).
4. Upload the WMS FDS `.docx` when prompted.
5. Mount Google Drive when prompted for persistent memory and observations.
6. Historical demonstration snapshots are simulated and clearly labelled.

## 1. Data loading and preparation
Load the training workbook and consolidate inventory, demand and active replenishment tasks.

In [1]:
from google.colab import files

uploaded = files.upload()

Saving AI_Warehouse_Intelligence_Training_Dataset.xlsx to AI_Warehouse_Intelligence_Training_Dataset.xlsx


In [2]:
import pandas as pd

file_name = next(iter(uploaded))

inventory_df = pd.read_excel(
    file_name,
    sheet_name="Inventory_Snapshot"
)

demand_df = pd.read_excel(
    file_name,
    sheet_name="Demand_History_30_Days"
)

tasks_df = pd.read_excel(
    file_name,
    sheet_name="Replenishment_Tasks"
)

print("Inventory Data:")
display(inventory_df.head())

print("Demand History:")
display(demand_df.head())

print("Replenishment Tasks:")
display(tasks_df.head())

Inventory Data:


,SKU,Velocity_Class,Forward_Pick_Qty,Reserve_Qty,Minimum_Qty,Maximum_Qty,Avg_Daily_Demand,Open_Customer_Demand,Replenishment_Lead_Time_Days,Forward_Pick_Location,SKU_Status
0,SKU1001,Fast,25,200,30,100,35,45,1,A-01-01,Active
1,SKU1002,Medium,80,150,40,120,20,15,2,A-02-03,Active
2,SKU1003,Fast,15,50,25,80,40,60,2,B-01-02,Active
3,SKU1004,Slow,45,300,30,100,15,10,3,C-01-04,Active
4,SKU1005,Fast,20,0,20,90,30,35,1,A-03-01,Active


Demand History:


,Date,SKU,Daily_Demand
0,2026-10-01,SKU1001,39
1,2026-10-02,SKU1001,42
2,2026-10-03,SKU1001,31
3,2026-10-04,SKU1001,34
4,2026-10-05,SKU1001,37


Replenishment Tasks:


,Task_ID,SKU,Source_Area,Destination_Location,Task_Qty,Completed_Qty,Task_Status,Created_Timestamp
0,REP001,SKU1001,RESERVE,A-01-01,30,10,IN_PROGRESS,2026-11-11 09:15
1,REP002,SKU1002,RESERVE,A-02-03,20,20,COMPLETED,2026-11-10 14:20
2,REP003,SKU1003,RESERVE,B-01-02,25,25,OPEN,2026-11-11 08:40
3,REP004,SKU1006,RESERVE,B-02-01,40,15,IN_PROGRESS,2026-11-11 10:05
4,REP005,SKU1007,RESERVE,A-04-02,18,0,OPEN,2026-11-11 11:10


In [3]:
# Calculate remaining task quantity

tasks_df["Remaining_Qty"] = (
    tasks_df["Task_Qty"] -
    tasks_df["Completed_Qty"]
).clip(lower=0)

# Identify eligible active tasks

active_tasks = tasks_df[
    tasks_df["Task_Status"].isin(
        ["OPEN", "IN_PROGRESS"]
    )
]

# Calculate total remaining quantity for each SKU

open_task_summary = (
    active_tasks.groupby("SKU")["Remaining_Qty"]
    .sum()
    .reset_index()
)

print("Active Replenishment Summary:")
display(open_task_summary)

Active Replenishment Summary:


,SKU,Remaining_Qty
0,SKU1001,20
1,SKU1003,0
2,SKU1006,25
3,SKU1007,18


In [4]:
warehouse_df = inventory_df.merge(
    open_task_summary,
    on="SKU",
    how="left"
)

warehouse_df["Remaining_Qty"] = (
    warehouse_df["Remaining_Qty"].fillna(0)
)

print("Consolidated Warehouse Data:")
display(warehouse_df)

Consolidated Warehouse Data:


,SKU,Velocity_Class,Forward_Pick_Qty,Reserve_Qty,Minimum_Qty,Maximum_Qty,Avg_Daily_Demand,Open_Customer_Demand,Replenishment_Lead_Time_Days,Forward_Pick_Location,SKU_Status,Remaining_Qty
0,SKU1001,Fast,25,200,30,100,35,45,1,A-01-01,Active,20.0
1,SKU1002,Medium,80,150,40,120,20,15,2,A-02-03,Active,0.0
2,SKU1003,Fast,15,50,25,80,40,60,2,B-01-02,Active,0.0
3,SKU1004,Slow,45,300,30,100,15,10,3,C-01-04,Active,0.0
4,SKU1005,Fast,20,0,20,90,30,35,1,A-03-01,Active,0.0
5,SKU1006,Medium,55,120,35,110,25,40,2,B-02-01,Active,25.0
6,SKU1007,Fast,12,18,25,100,50,75,1,A-04-02,Active,18.0
7,SKU1008,Slow,90,400,50,150,8,5,4,C-02-02,Active,0.0
8,SKU1009,Medium,32,70,35,100,28,20,2,B-03-03,Active,0.0
9,SKU1010,Fast,40,25,30,120,45,55,3,A-05-01,Active,0.0


In [5]:
# Calculate demand statistics for every SKU

demand_summary = (
    demand_df.groupby("SKU")
    .agg(
        Total_Demand=("Daily_Demand", "sum"),
        Average_Daily_Demand=("Daily_Demand", "mean"),
        Maximum_Daily_Demand=("Daily_Demand", "max")
    )
    .reset_index()
)

demand_summary["Average_Daily_Demand"] = (
    demand_summary["Average_Daily_Demand"].round(2)
)

print("SKU Demand Analysis:")
display(demand_summary)

SKU Demand Analysis:


,SKU,Total_Demand,Average_Daily_Demand,Maximum_Daily_Demand
0,SKU1001,1070,35.67,42
1,SKU1002,653,21.77,29
2,SKU1003,1250,41.67,49
3,SKU1004,470,15.67,22
4,SKU1005,953,31.77,39
5,SKU1006,809,26.97,34
6,SKU1007,1430,47.67,51
7,SKU1008,284,9.47,17
8,SKU1009,890,29.67,37
9,SKU1010,1370,45.67,52


In [6]:
# Combine demand statistics with inventory information

analytics_df = warehouse_df.merge(
    demand_summary,
    on="SKU",
    how="left"
)

# Calculate current forward-pick days of cover

analytics_df["Forward_Pick_Days_Cover"] = (
    analytics_df["Forward_Pick_Qty"] /
    analytics_df["Average_Daily_Demand"].replace(0, float("nan"))
).round(2)

# Calculate projected forward-pick stock,
# including remaining active replenishment tasks

analytics_df["Projected_Forward_Pick"] = (
    analytics_df["Forward_Pick_Qty"] +
    analytics_df["Remaining_Qty"]
)

analytics_df["Projected_Days_Cover"] = (
    analytics_df["Projected_Forward_Pick"] /
    analytics_df["Average_Daily_Demand"].replace(0, float("nan"))
).round(2)

print("Consolidated Warehouse Analytics:")
display(analytics_df)

Consolidated Warehouse Analytics:


,SKU,Velocity_Class,Forward_Pick_Qty,Reserve_Qty,Minimum_Qty,Maximum_Qty,Avg_Daily_Demand,Open_Customer_Demand,Replenishment_Lead_Time_Days,Forward_Pick_Location,SKU_Status,Remaining_Qty,Total_Demand,Average_Daily_Demand,Maximum_Daily_Demand,Forward_Pick_Days_Cover,Projected_Forward_Pick,Projected_Days_Cover
0,SKU1001,Fast,25,200,30,100,35,45,1,A-01-01,Active,20.0,1070,35.67,42,0.70,45.0,1.26
1,SKU1002,Medium,80,150,40,120,20,15,2,A-02-03,Active,0.0,653,21.77,29,3.67,80.0,3.67
2,SKU1003,Fast,15,50,25,80,40,60,2,B-01-02,Active,0.0,1250,41.67,49,0.36,15.0,0.36
3,SKU1004,Slow,45,300,30,100,15,10,3,C-01-04,Active,0.0,470,15.67,22,2.87,45.0,2.87
4,SKU1005,Fast,20,0,20,90,30,35,1,A-03-01,Active,0.0,953,31.77,39,0.63,20.0,0.63
5,SKU1006,Medium,55,120,35,110,25,40,2,B-02-01,Active,25.0,809,26.97,34,2.04,80.0,2.97
6,SKU1007,Fast,12,18,25,100,50,75,1,A-04-02,Active,18.0,1430,47.67,51,0.25,30.0,0.63
7,SKU1008,Slow,90,400,50,150,8,5,4,C-02-02,Active,0.0,284,9.47,17,9.50,90.0,9.50
8,SKU1009,Medium,32,70,35,100,28,20,2,B-03-03,Active,0.0,890,29.67,37,1.08,32.0,1.08
9,SKU1010,Fast,40,25,30,120,45,55,3,A-05-01,Active,0.0,1370,45.67,52,0.88,40.0,0.88


In [7]:
# Classify SKU movement based on average daily demand

high_demand_limit = analytics_df[
    "Average_Daily_Demand"
].quantile(0.70)

low_demand_limit = analytics_df[
    "Average_Daily_Demand"
].quantile(0.30)


def classify_movement(demand):
    if demand >= high_demand_limit:
        return "FAST MOVING"
    elif demand <= low_demand_limit:
        return "SLOW MOVING"
    else:
        return "MEDIUM MOVING"


analytics_df["SKU_Movement"] = (
    analytics_df["Average_Daily_Demand"]
    .apply(classify_movement)
)


# Identify replenishment triggers

analytics_df["Replenishment_Trigger"] = (
    analytics_df["Forward_Pick_Qty"] <
    analytics_df["Minimum_Qty"]
)


# Identify potential stockout risk

def stockout_risk(days):

    if pd.isna(days):
        return "DEMAND DATA MISSING"

    elif days <= 1:
        return "HIGH RISK"

    elif days <= 2:
        return "MEDIUM RISK"

    else:
        return "LOW RISK"


analytics_df["Stockout_Risk"] = (
    analytics_df["Projected_Days_Cover"]
    .apply(stockout_risk)
)

print("Stockout risk classification completed!")


print("AI Warehouse Analytics Report:")

display(
    analytics_df[[
        "SKU",
        "Forward_Pick_Qty",
        "Reserve_Qty",
        "Average_Daily_Demand",
        "SKU_Movement",
        "Forward_Pick_Days_Cover",
        "Projected_Days_Cover",
        "Replenishment_Trigger",
        "Stockout_Risk"

    ]]
)

Stockout risk classification completed!
AI Warehouse Analytics Report:


,SKU,Forward_Pick_Qty,Reserve_Qty,Average_Daily_Demand,SKU_Movement,Forward_Pick_Days_Cover,Projected_Days_Cover,Replenishment_Trigger,Stockout_Risk
0,SKU1001,25,200,35.67,MEDIUM MOVING,0.70,1.26,True,MEDIUM RISK
1,SKU1002,80,150,21.77,SLOW MOVING,3.67,3.67,False,LOW RISK
2,SKU1003,15,50,41.67,FAST MOVING,0.36,0.36,True,HIGH RISK
3,SKU1004,45,300,15.67,SLOW MOVING,2.87,2.87,False,LOW RISK
4,SKU1005,20,0,31.77,MEDIUM MOVING,0.63,0.63,False,HIGH RISK
5,SKU1006,55,120,26.97,MEDIUM MOVING,2.04,2.97,False,LOW RISK
6,SKU1007,12,18,47.67,FAST MOVING,0.25,0.63,True,HIGH RISK
7,SKU1008,90,400,9.47,SLOW MOVING,9.50,9.50,False,LOW RISK
8,SKU1009,32,70,29.67,MEDIUM MOVING,1.08,1.08,True,MEDIUM RISK
9,SKU1010,40,25,45.67,FAST MOVING,0.88,0.88,False,HIGH RISK


## 2. Demand forecasting
Prepare lag and rolling-demand features, evaluate a Random Forest model against a simple baseline, and generate a seven-day SKU forecast. Forecast quality is specific to this simulated dataset.

In [8]:
import pandas as pd
import numpy as np

# Ensure dates are in the correct format
demand_df["Date"] = pd.to_datetime(demand_df["Date"])

# Sort demand records SKU-wise and date-wise
demand_df = demand_df.sort_values(
    ["SKU", "Date"]
).copy()

# Previous day's demand
demand_df["Lag_1"] = (
    demand_df.groupby("SKU")["Daily_Demand"].shift(1)
)

# Demand from 7 days ago
demand_df["Lag_7"] = (
    demand_df.groupby("SKU")["Daily_Demand"].shift(7)
)

# Average demand over the previous 7 days
demand_df["Rolling_7"] = (
    demand_df.groupby("SKU")["Daily_Demand"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

# Day of the week
demand_df["Day_of_Week"] = (
    demand_df["Date"].dt.dayofweek
)

# Remove records without sufficient historical information
forecast_df = demand_df.dropna().copy()

print("Prepared Forecasting Dataset:")
display(forecast_df.head(10))

print("Total usable records:", len(forecast_df))

Prepared Forecasting Dataset:


,Date,SKU,Daily_Demand,Lag_1,Lag_7,Rolling_7,Day_of_Week
7,2026-10-08,SKU1001,42,34.0,39.0,35.428571,3
8,2026-10-09,SKU1001,36,42.0,42.0,35.857143,4
9,2026-10-10,SKU1001,34,36.0,31.0,35.000000,5
10,2026-10-11,SKU1001,37,34.0,34.0,35.428571,6
11,2026-10-12,SKU1001,31,37.0,37.0,35.857143,0
12,2026-10-13,SKU1001,34,31.0,31.0,35.000000,1
13,2026-10-14,SKU1001,37,34.0,34.0,35.428571,2
14,2026-10-15,SKU1001,36,37.0,42.0,35.857143,3
15,2026-10-16,SKU1001,39,36.0,36.0,35.000000,4
16,2026-10-17,SKU1001,37,39.0,34.0,35.428571,5


Total usable records: 230


In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Select model inputs
features = [
    "SKU",
    "Day_of_Week",
    "Lag_1",
    "Lag_7",
    "Rolling_7"
]

X = forecast_df[features]
y = forecast_df["Daily_Demand"]

# Convert SKU names into numerical model inputs
X = pd.get_dummies(X, columns=["SKU"], dtype=int)

# Use the last 7 dates for testing
cutoff_date = (
    forecast_df["Date"].max() - pd.Timedelta(days=6)
)

train_mask = forecast_df["Date"] < cutoff_date
test_mask = forecast_df["Date"] >= cutoff_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

# Train the model
warehouse_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

warehouse_model.fit(X_train, y_train)

# Predict test-period demand
test_predictions = warehouse_model.predict(X_test)

# Evaluate the model
model_mae = mean_absolute_error(
    y_test,
    test_predictions
)

# Compare against a simple 7-day average baseline
baseline_mae = mean_absolute_error(
    y_test,
    forecast_df.loc[test_mask, "Rolling_7"]
)

print("Random Forest MAE:", round(model_mae, 2))
print("7-Day Average Baseline MAE:", round(baseline_mae, 2))
print("Training Records:", len(X_train))
print("Testing Records:", len(X_test))

Random Forest MAE: 1.7
7-Day Average Baseline MAE: 3.11
Training Records: 160
Testing Records: 70


In [10]:
# Get all SKUs
sku_list = sorted(demand_df["SKU"].unique())

# Keep historical demand for each SKU
history = {
    sku: list(
        demand_df[demand_df["SKU"] == sku]
        ["Daily_Demand"]
    )
    for sku in sku_list
}

# Generate the next 7 dates
future_dates = pd.date_range(
    start=demand_df["Date"].max() + pd.Timedelta(days=1),
    periods=7,
    freq="D"
)

forecast_results = []

# Generate forecasts one day at a time
for future_date in future_dates:

    for sku in sku_list:

        previous_values = history[sku]

        input_data = pd.DataFrame([{
            "SKU": sku,
            "Day_of_Week": future_date.dayofweek,
            "Lag_1": previous_values[-1],
            "Lag_7": previous_values[-7],
            "Rolling_7": np.mean(previous_values[-7:])
        }])

        # Apply the same SKU encoding as the trained model
        input_data = pd.get_dummies(
            input_data,
            columns=["SKU"],
            dtype=int
        )

        input_data = input_data.reindex(
            columns=X_train.columns,
            fill_value=0
        )

        predicted_demand = max(
            0,
            round(warehouse_model.predict(input_data)[0])
        )

        forecast_results.append({
            "Date": future_date,
            "SKU": sku,
            "Predicted_Demand": predicted_demand
        })

        # Add prediction to history for the next day's forecast
        history[sku].append(predicted_demand)


future_forecast_df = pd.DataFrame(forecast_results)

print("Next 7 Days Demand Forecast:")
display(future_forecast_df)

Next 7 Days Demand Forecast:


,Date,SKU,Predicted_Demand
0,2026-10-31,SKU1001,34
1,2026-10-31,SKU1002,22
2,2026-10-31,SKU1003,40
3,2026-10-31,SKU1004,16
4,2026-10-31,SKU1005,32
...,...,...,...
65,2026-11-06,SKU1006,27
66,2026-11-06,SKU1007,51
67,2026-11-06,SKU1008,12
68,2026-11-06,SKU1009,32


In [11]:
forecast_file = "AI_Warehouse_Demand_Forecast.xlsx"

with pd.ExcelWriter(forecast_file) as writer:

    future_forecast_df.to_excel(
        writer,
        sheet_name="7_Day_Forecast",
        index=False
    )

    future_forecast_df.groupby("SKU")[
        "Predicted_Demand"
    ].sum().reset_index().to_excel(
        writer,
        sheet_name="SKU_7_Day_Total",
        index=False
    )

print("Forecast report created:", forecast_file)

Forecast report created: AI_Warehouse_Demand_Forecast.xlsx


## 3. Replenishment recommendation engine
Estimate lead-time demand, account for active replenishment tasks, and calculate a recommendation using the current project rules. Review the outputs as recommendations, not executed WMS tasks.

In [12]:


# Identify the lead-time column
if "Replenishment_Lead_Time_Days" in inventory_df.columns:
    lead_time_column = "Replenishment_Lead_Time_Days"
elif "Lead_Time_Days" in inventory_df.columns:
    lead_time_column = "Lead_Time_Days"
else:
    raise ValueError("Lead-time column not found. Check inventory_df.columns")

replenishment_results = []

for _, item in inventory_df.iterrows():

    sku = item["SKU"]
    current_fp = item["Forward_Pick_Qty"]
    minimum_qty = item["Minimum_Qty"]
    target_qty = item["Maximum_Qty"]
    reserve_qty = item["Reserve_Qty"]
    lead_time = int(item[lead_time_column])

    # Limit lead time to the available 7-day forecast
    lead_time = max(1, min(lead_time, 7))

    # Get this SKU's forecast
    sku_forecast = future_forecast_df[
        future_forecast_df["SKU"] == sku
    ].sort_values("Date")

    # Demand expected during replenishment lead time
    lead_time_demand = sku_forecast.head(lead_time)[
        "Predicted_Demand"
    ].sum()

    # Inventory projected after lead-time demand
    projected_fp = max(0, current_fp - lead_time_demand)

    # Find remaining quantity in active replenishment tasks
    open_task_qty = warehouse_df.loc[
        warehouse_df["SKU"] == sku, "Remaining_Qty"
    ].iloc[0]

    # Projected position if existing tasks are completed
    projected_with_tasks = projected_fp + open_task_qty

    replenishment_results.append({
        "SKU": sku,
        "Current_Forward_Pick": current_fp,
        "Minimum_Qty": minimum_qty,
        "Target_Qty": target_qty,
        "Reserve_Qty": reserve_qty,
        "Lead_Time_Days": lead_time,
        "Forecast_Lead_Time_Demand": lead_time_demand,
        "Projected_FP_Before_Tasks": projected_fp,
        "Existing_Open_Task_Qty": open_task_qty,
        "Projected_FP_With_Tasks": projected_with_tasks
    })

replenishment_df = pd.DataFrame(replenishment_results)

display(replenishment_df)




,SKU,Current_Forward_Pick,Minimum_Qty,Target_Qty,Reserve_Qty,Lead_Time_Days,Forecast_Lead_Time_Demand,Projected_FP_Before_Tasks,Existing_Open_Task_Qty,Projected_FP_With_Tasks
0,SKU1001,25,30,100,200,1,34,0,20.0,20.0
1,SKU1002,80,40,120,150,2,40,40,0.0,40.0
2,SKU1003,15,25,80,50,2,83,0,0.0,0.0
3,SKU1004,45,30,100,300,3,43,2,0.0,2.0
4,SKU1005,20,20,90,0,1,32,0,0.0,0.0
5,SKU1006,55,35,110,120,2,50,5,25.0,30.0
6,SKU1007,12,25,100,18,1,46,0,18.0,18.0
7,SKU1008,90,50,150,400,4,49,41,0.0,41.0
8,SKU1009,32,35,100,70,2,57,0,0.0,0.0
9,SKU1010,40,30,120,25,3,132,0,0.0,0.0


In [13]:
# Prepare input for the final replenishment engine

final_report = replenishment_df.copy()

print("INPUT DATA READY FOR FINAL REPLENISHMENT ENGINE")

display(final_report.head())

INPUT DATA READY FOR FINAL REPLENISHMENT ENGINE


,SKU,Current_Forward_Pick,Minimum_Qty,Target_Qty,Reserve_Qty,Lead_Time_Days,Forecast_Lead_Time_Demand,Projected_FP_Before_Tasks,Existing_Open_Task_Qty,Projected_FP_With_Tasks
0,SKU1001,25,30,100,200,1,34,0,20.0,20.0
1,SKU1002,80,40,120,150,2,40,40,0.0,40.0
2,SKU1003,15,25,80,50,2,83,0,0.0,0.0
3,SKU1004,45,30,100,300,3,43,2,0.0,2.0
4,SKU1005,20,20,90,0,1,32,0,0.0,0.0


In [14]:

import pandas as pd

# ============================================================
# FINAL CONSOLIDATED REPLENISHMENT ENGINE
# Processes ALL SKUs
# ============================================================

def run_replenishment_engine(final_report, tasks_df):

    # 1. Validate required inventory and forecast columns
    required_report_columns = [
        "SKU",
        "Current_Forward_Pick",
        "Minimum_Qty",
        "Target_Qty",
        "Reserve_Qty",
        "Forecast_Lead_Time_Demand"
    ]

    missing_columns = [
        col for col in required_report_columns
        if col not in final_report.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in final_report: {missing_columns}"
        )

    # 2. Validate required task columns
    required_task_columns = [
        "SKU",
        "Task_Qty",
        "Completed_Qty",
        "Task_Status",
        "Source_Area"
    ]

    missing_task_columns = [
        col for col in required_task_columns
        if col not in tasks_df.columns
    ]

    if missing_task_columns:
        raise ValueError(
            f"Missing columns in tasks_df: {missing_task_columns}"
        )

    # 3. Prepare task data
    tasks = tasks_df.copy()

    tasks["Task_Status"] = (
        tasks["Task_Status"].astype(str).str.strip().str.upper()
    )

    tasks["Source_Area"] = (
        tasks["Source_Area"].astype(str).str.strip().str.upper()
    )

    tasks["Remaining_Qty"] = (
        pd.to_numeric(tasks["Task_Qty"], errors="coerce").fillna(0)
        - pd.to_numeric(tasks["Completed_Qty"], errors="coerce").fillna(0)
    ).clip(lower=0)

    # Consider only active tasks originating from RESERVE
    active_reserve_tasks = tasks[
        (tasks["Task_Status"].isin(["OPEN", "IN_PROGRESS"]))
        & (tasks["Source_Area"] == "RESERVE")
    ]

    # 4. Process every SKU
    recommendation_results = []

    for _, row in final_report.iterrows():

        sku = row["SKU"]

        current_fp = float(row["Current_Forward_Pick"])
        minimum_qty = float(row["Minimum_Qty"])
        target_qty = float(row["Target_Qty"])
        reserve_qty = float(row["Reserve_Qty"])
        forecast_demand = float(row["Forecast_Lead_Time_Demand"])

        # 5. Identify active reserve replenishment tasks for this SKU
        sku_tasks = active_reserve_tasks[
            active_reserve_tasks["SKU"] == sku
        ]

        existing_open_task_qty = float(
            sku_tasks["Remaining_Qty"].sum()
        )

        # 6. Calculate reserve stock committed to existing tasks
        reserve_committed_qty = existing_open_task_qty

        eligible_available_reserve = max(
            0,
            reserve_qty - reserve_committed_qty
        )

        # 7. Project Forward Pick stock after lead-time demand
        projected_fp_before_tasks = max(
            0,
            current_fp - forecast_demand
        )

        # Identify demand that exceeds current Forward Pick stock
        uncovered_lead_time_demand = max(
            0,
            forecast_demand - current_fp
        )

        # 8. Include existing replenishment task quantities
        projected_fp_with_tasks = (
            projected_fp_before_tasks
            + existing_open_task_qty
        )

        # 9. Check whether replenishment is required
        replenishment_trigger = (
            current_fp < minimum_qty
            or projected_fp_before_tasks <= minimum_qty
        )

        # 10. Calculate remaining quantity required to reach target
        net_required_qty = max(
            0,
            target_qty - projected_fp_with_tasks
        )

        # 11. Calculate recommendation and shortage
        if not replenishment_trigger:

            recommended_task_qty = 0
            unfulfilled_qty = 0
            task_action = "NO REPLENISHMENT REQUIRED"

        elif net_required_qty == 0:

            recommended_task_qty = 0
            unfulfilled_qty = 0
            task_action = "NO ADDITIONAL REPLENISHMENT REQUIRED"

        elif existing_open_task_qty > 0:

            recommended_task_qty = 0
            unfulfilled_qty = 0
            task_action = "REVIEW EXISTING OPEN TASK"

        else:

            recommended_task_qty = min(
                net_required_qty,
                eligible_available_reserve
            )

            unfulfilled_qty = max(
                0,
                net_required_qty - recommended_task_qty
            )

            if unfulfilled_qty > 0 and recommended_task_qty > 0:
                task_action = "PARTIAL REPLENISHMENT - RESERVE SHORTAGE"

            elif recommended_task_qty > 0:
                task_action = "CREATE REPLENISHMENT TASK"

            else:
                task_action = "RAISE RESERVE SHORTAGE EXCEPTION"

        # 12. Store results for this SKU
        recommendation_results.append({

            "SKU": sku,

            "Current_Forward_Pick": current_fp,
            "Minimum_Qty": minimum_qty,
            "Target_Qty": target_qty,
            "Forecast_Lead_Time_Demand": forecast_demand,

            "Uncovered_Lead_Time_Demand": uncovered_lead_time_demand,

            "Projected_FP_Before_Tasks": projected_fp_before_tasks,
            "Existing_Open_Task_Qty": existing_open_task_qty,
            "Projected_FP_With_Tasks": projected_fp_with_tasks,

            "Replenishment_Trigger": replenishment_trigger,
            "Net_Required_Qty": net_required_qty,

            "Gross_Reserve_Qty": reserve_qty,
            "Reserve_Committed_Qty": reserve_committed_qty,
            "Eligible_Available_Reserve": eligible_available_reserve,

            "Recommended_Task_Qty": recommended_task_qty,
            "Unfulfilled_Qty": unfulfilled_qty,

            "Task_Action": task_action
        })

    # 13. Create final recommendation DataFrame
    recommendation_df = pd.DataFrame(recommendation_results)

    return recommendation_df


# ============================================================
# RUN THE FINAL ENGINE
# ============================================================

replenishment_recommendation_df = run_replenishment_engine(
    final_report,
    tasks_df
)

print("=" * 80)
print("FINAL AI WAREHOUSE REPLENISHMENT RECOMMENDATIONS")
print("=" * 80)

display(
    replenishment_recommendation_df.sort_values(
        by=["Replenishment_Trigger", "Unfulfilled_Qty"],
        ascending=[False, False]
    )
)

print("\nTOTAL SKUs PROCESSED:",
      len(replenishment_recommendation_df))

print("\nREPLENISHMENT ACTION SUMMARY:")

display(
    replenishment_recommendation_df[
        "Task_Action"
    ].value_counts().rename_axis("Task_Action").reset_index(
        name="SKU_Count"
    )
)


FINAL AI WAREHOUSE REPLENISHMENT RECOMMENDATIONS


,SKU,Current_Forward_Pick,Minimum_Qty,Target_Qty,Forecast_Lead_Time_Demand,Uncovered_Lead_Time_Demand,Projected_FP_Before_Tasks,Existing_Open_Task_Qty,Projected_FP_With_Tasks,Replenishment_Trigger,Net_Required_Qty,Gross_Reserve_Qty,Reserve_Committed_Qty,Eligible_Available_Reserve,Recommended_Task_Qty,Unfulfilled_Qty,Task_Action
9,SKU1010,40.0,30.0,120.0,132.0,92.0,0.0,0.0,0.0,True,120.0,25.0,0.0,25.0,25.0,95.0,PARTIAL REPLENISHMENT - RESERVE SHORTAGE
4,SKU1005,20.0,20.0,90.0,32.0,12.0,0.0,0.0,0.0,True,90.0,0.0,0.0,0.0,0.0,90.0,RAISE RESERVE SHORTAGE EXCEPTION
2,SKU1003,15.0,25.0,80.0,83.0,68.0,0.0,0.0,0.0,True,80.0,50.0,0.0,50.0,50.0,30.0,PARTIAL REPLENISHMENT - RESERVE SHORTAGE
8,SKU1009,32.0,35.0,100.0,57.0,25.0,0.0,0.0,0.0,True,100.0,70.0,0.0,70.0,70.0,30.0,PARTIAL REPLENISHMENT - RESERVE SHORTAGE
0,SKU1001,25.0,30.0,100.0,34.0,9.0,0.0,20.0,20.0,True,80.0,200.0,20.0,180.0,0.0,0.0,REVIEW EXISTING OPEN TASK
1,SKU1002,80.0,40.0,120.0,40.0,0.0,40.0,0.0,40.0,True,80.0,150.0,0.0,150.0,80.0,0.0,CREATE REPLENISHMENT TASK
3,SKU1004,45.0,30.0,100.0,43.0,0.0,2.0,0.0,2.0,True,98.0,300.0,0.0,300.0,98.0,0.0,CREATE REPLENISHMENT TASK
5,SKU1006,55.0,35.0,110.0,50.0,0.0,5.0,25.0,30.0,True,80.0,120.0,25.0,95.0,0.0,0.0,REVIEW EXISTING OPEN TASK
6,SKU1007,12.0,25.0,100.0,46.0,34.0,0.0,18.0,18.0,True,82.0,18.0,18.0,0.0,0.0,0.0,REVIEW EXISTING OPEN TASK
7,SKU1008,90.0,50.0,150.0,49.0,0.0,41.0,0.0,41.0,True,109.0,400.0,0.0,400.0,109.0,0.0,CREATE REPLENISHMENT TASK



TOTAL SKUs PROCESSED: 10

REPLENISHMENT ACTION SUMMARY:


,Task_Action,SKU_Count
0,REVIEW EXISTING OPEN TASK,3
1,CREATE REPLENISHMENT TASK,3
2,PARTIAL REPLENISHMENT - RESERVE SHORTAGE,3
3,RAISE RESERVE SHORTAGE EXCEPTION,1


### SKU1007 validation scenario
Let's take a particular SKU validation for SKU1007 that checks open reserve tasks and eligible reserve availability. Keep this as a focused validation example; the calculation depends on the assumptions documented in the code.

In [15]:
# CORRECTED REPLENISHMENT ENGINE
# SKU-level validation for SKU1007

import pandas as pd

sku = "SKU1007"

# 1. Get SKU details
sku_data = replenishment_df[
    replenishment_df["SKU"] == sku
].iloc[0]

current_fp = float(sku_data["Current_Forward_Pick"])
target_qty = float(sku_data["Target_Qty"])
reserve_qty = float(sku_data["Reserve_Qty"])
forecast_demand = float(
    sku_data["Forecast_Lead_Time_Demand"]
)

# 2. Identify active replenishment tasks
sku_tasks = tasks_df[
    tasks_df["SKU"] == sku
].copy()

sku_tasks["Remaining_Qty"] = (
    sku_tasks["Task_Qty"]
    - sku_tasks["Completed_Qty"]
).clip(lower=0)

active_tasks = sku_tasks[
    sku_tasks["Task_Status"].isin(
        ["OPEN", "IN_PROGRESS"]
    )
]

# 3. Calculate existing open task quantity
existing_open_task_qty = (
    active_tasks["Remaining_Qty"].sum()
)

# 4. Calculate reserve stock committed
# Assumption: Reserve_Qty is gross stock,
# before open task allocations.

reserve_tasks = active_tasks[
    active_tasks["Source_Area"] == "RESERVE"
]

reserve_committed_qty = (
    reserve_tasks["Remaining_Qty"].sum()
)

eligible_available_reserve = max(
    0,
    reserve_qty - reserve_committed_qty
)

# 5. Project FP inventory after lead-time demand
projected_fp_before_tasks = max(
    0,
    current_fp - forecast_demand
)

# 6. Add existing open replenishment tasks
projected_fp_with_tasks = (
    projected_fp_before_tasks
    + existing_open_task_qty
)

# 7. Calculate remaining requirement
net_required_qty = max(
    0,
    target_qty - projected_fp_with_tasks
)

# 8. Calculate new task quantity
recommended_task_qty = min(
    net_required_qty,
    eligible_available_reserve
)

# 9. Calculate shortage
unfulfilled_qty = max(
    0,
    net_required_qty - recommended_task_qty
)

# 10. Determine action
if net_required_qty == 0:

    task_action = "NO REPLENISHMENT REQUIRED"

elif existing_open_task_qty > 0:

    if unfulfilled_qty > 0:
        task_action = (
            "REVIEW EXISTING OPEN TASK "
            "+ RESERVE SHORTAGE"
        )
    else:
        task_action = "REVIEW EXISTING OPEN TASK"

elif recommended_task_qty > 0 and unfulfilled_qty > 0:

    task_action = (
        "CREATE PARTIAL TASK + "
        "RAISE SHORTAGE EXCEPTION"
    )

elif recommended_task_qty > 0:

    task_action = "CREATE REPLENISHMENT TASK"

else:

    task_action = "RAISE RESERVE SHORTAGE EXCEPTION"

# Display results
print("=" * 65)
print("CORRECTED REPLENISHMENT ENGINE")
print("=" * 65)

print("SKU:", sku)
print("Current Forward Pick:", current_fp)
print("Forecast Lead-Time Demand:", forecast_demand)
print("Projected FP Before Tasks:", projected_fp_before_tasks)
print("Existing Open Task Qty:", existing_open_task_qty)
print("Projected FP With Tasks:", projected_fp_with_tasks)
print("Target Qty:", target_qty)

print("-" * 65)

print("Net Required Qty:", net_required_qty)
print("Gross Reserve Qty:", reserve_qty)
print("Reserve Committed to Tasks:", reserve_committed_qty)
print("Eligible Available Reserve:", eligible_available_reserve)
print("Recommended New Task Qty:", recommended_task_qty)
print("Unfulfilled Qty:", unfulfilled_qty)

print("-" * 65)
print("Task Action:", task_action)
print("=" * 65)

CORRECTED REPLENISHMENT ENGINE
SKU: SKU1007
Current Forward Pick: 12.0
Forecast Lead-Time Demand: 46.0
Projected FP Before Tasks: 0
Existing Open Task Qty: 18
Projected FP With Tasks: 18
Target Qty: 100.0
-----------------------------------------------------------------
Net Required Qty: 82.0
Gross Reserve Qty: 18.0
Reserve Committed to Tasks: 18
Eligible Available Reserve: 0
Recommended New Task Qty: 0
Unfulfilled Qty: 82.0
-----------------------------------------------------------------
Task Action: REVIEW EXISTING OPEN TASK + RESERVE SHORTAGE


## 4. Python warehouse tools
These functions expose inventory status, stockout risk and replenishment recommendations to the AI agent.

In [16]:
# Tool 1: Get inventory and operational status of a SKU


def get_sku_status(sku):

    # Get the SKU's inventory and demand information
    result = analytics_df[
        analytics_df["SKU"].str.upper() == sku.upper()
    ]

    if result.empty:
        return {"message": f"SKU {sku} not found"}

    # Convert inventory details into a dictionary
    status = result.iloc[0].to_dict()

    # Get the authoritative replenishment recommendation
    replenishment = replenishment_recommendation_df[
        replenishment_recommendation_df["SKU"].str.upper()
        == sku.upper()
    ]

    if replenishment.empty:
        return {"message": f"Replenishment data not found for {sku}"}

    replenishment = replenishment.iloc[0]

    # Use the final engine's projected inventory
    projected_fp = replenishment["Projected_FP_With_Tasks"]

    average_demand = status["Average_Daily_Demand"]

    # Calculate projected days of cover consistently
    if average_demand > 0:
        projected_days_cover = projected_fp / average_demand
    else:
        projected_days_cover = None

    # Replace old projection values
    status["Projected_Forward_Pick"] = projected_fp
    status["Projected_Days_Cover"] = projected_days_cover

    # Update replenishment information from final engine
    for column in [
        "Existing_Open_Task_Qty",
        "Net_Required_Qty",
        "Gross_Reserve_Qty",
        "Reserve_Committed_Qty",
        "Eligible_Available_Reserve",
        "Recommended_Task_Qty",
        "Unfulfilled_Qty",
        "Task_Action"
    ]:
        status[column] = replenishment[column]

    # Recalculate stockout risk using the canonical projection
    if projected_days_cover is None:
        status["Stockout_Risk"] = "DEMAND DATA MISSING"
    elif projected_days_cover <= 1:
        status["Stockout_Risk"] = "HIGH RISK"
    elif projected_days_cover <= 2:
        status["Stockout_Risk"] = "MEDIUM RISK"
    else:
        status["Stockout_Risk"] = "LOW RISK"

    return status



# Tool 2: Identify high stockout-risk SKUs


def get_stockout_risks():

    result = analytics_df.copy()

    # Use canonical replenishment projections
    for _, row in replenishment_recommendation_df.iterrows():

        sku = row["SKU"]

        mask = result["SKU"] == sku

        projected_fp = row["Projected_FP_With_Tasks"]

        result.loc[mask, "Projected_Forward_Pick"] = projected_fp

    # Recalculate projected days of cover
    result["Projected_Days_Cover"] = (
        result["Projected_Forward_Pick"]
        / result["Average_Daily_Demand"]
    )

    # Classify stockout risk
    result["Stockout_Risk"] = (
        result["Projected_Days_Cover"]
        .apply(stockout_risk)
    )

    # Return only high and medium risk SKUs
    result = result[
        result["Stockout_Risk"].isin(
            ["HIGH RISK", "MEDIUM RISK"]
        )
    ]

    return result[[
        "SKU",
        "Forward_Pick_Qty",
        "Projected_Forward_Pick",
        "Average_Daily_Demand",
        "Projected_Days_Cover",
        "Stockout_Risk"
    ]].to_dict(orient="records")



# Tool 3: Get replenishment recommendations

def get_replenishment_actions():

    return replenishment_recommendation_df[[
        "SKU",
        "Current_Forward_Pick",
        "Minimum_Qty",
        "Target_Qty",
        "Existing_Open_Task_Qty",
        "Net_Required_Qty",
        "Gross_Reserve_Qty",
        "Reserve_Committed_Qty",
        "Eligible_Available_Reserve",
        "Recommended_Task_Qty",
        "Unfulfilled_Qty",
        "Task_Action"
    ]].to_dict(orient="records")


print("Warehouse Copilot tools created successfully!")

Warehouse Copilot tools created successfully!


## 5. FDS knowledge retrieval (RAG)
Upload the Functional Design Specification, extract its paragraphs and tables, split it into chunks, and index the chunks using TF-IDF. The agent can retrieve relevant chunks through `search_fds`.

In [17]:
%pip install -q python-docx google-genai

In [18]:
from google.colab import files

uploaded_fds = files.upload()

Saving Forward_Pick_Replenishment_Seasonal_Peak_FDS.docx to Forward_Pick_Replenishment_Seasonal_Peak_FDS.docx


In [19]:
from docx import Document

file_name = next(iter(uploaded_fds))

doc = Document(file_name)

document_parts = []

# Extract paragraphs
for paragraph in doc.paragraphs:
    if paragraph.text.strip():
        document_parts.append(paragraph.text.strip())

# Extract tables
for table in doc.tables:
    for row in table.rows:
        cells = [cell.text.strip() for cell in row.cells]
        document_parts.append(" | ".join(cells))

fds_text = "\n".join(document_parts)

print("FDS successfully loaded!")
print("Total characters:", len(fds_text))
print("\nDocument preview:")
print(fds_text[:1500])

FDS successfully loaded!
Total characters: 21560

Document preview:
FORWARD-PICK REPLENISHMENT
DURING SEASONAL PEAKS
AS-IS Process | Problem Statement | TO-BE Process | Acceptance Criteria | Functional Design Specification
1. Purpose and scope
This document defines the current replenishment process, the operational problem, the proposed future-state process, testable acceptance criteria and the functional design for replenishing forward-pick locations before stock-outs occur during seasonal peaks. The design uses a threshold-based trigger, considers open demand and existing replenishment work, and prevents the creation of duplicate or over-sized tasks.
In scope: forward-pick inventory monitoring; min/target quantity; reserve eligibility; replenishment task creation and execution; exceptions; operational visibility.
Out of scope unless separately approved: purchasing, supplier planning, inbound appointment scheduling, labor optimization, and a full demand-forecasting implementation.
Sea

In [20]:
# Split the FDS into paragraphs
paragraphs = [
    p.strip()
    for p in fds_text.split("\n")
    if p.strip()
]

# Combine paragraphs into chunks
chunks = []
current_chunk = ""

CHUNK_SIZE = 1000

for paragraph in paragraphs:

    if len(current_chunk) + len(paragraph) <= CHUNK_SIZE:
        current_chunk += paragraph + "\n"

    else:
        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        current_chunk = paragraph + "\n"

if current_chunk.strip():
    chunks.append(current_chunk.strip())

print("FDS Chunking Completed!")
print("Total paragraphs:", len(paragraphs))
print("Total chunks:", len(chunks))

print("\nFirst chunk preview:\n")
print(chunks[0])

FDS Chunking Completed!
Total paragraphs: 182
Total chunks: 24

First chunk preview:

FORWARD-PICK REPLENISHMENT
DURING SEASONAL PEAKS
AS-IS Process | Problem Statement | TO-BE Process | Acceptance Criteria | Functional Design Specification
1. Purpose and scope
This document defines the current replenishment process, the operational problem, the proposed future-state process, testable acceptance criteria and the functional design for replenishing forward-pick locations before stock-outs occur during seasonal peaks. The design uses a threshold-based trigger, considers open demand and existing replenishment work, and prevents the creation of duplicate or over-sized tasks.
In scope: forward-pick inventory monitoring; min/target quantity; reserve eligibility; replenishment task creation and execution; exceptions; operational visibility.
Out of scope unless separately approved: purchasing, supplier planning, inbound appointment scheduling, labor optimization, and a full demand-forecasting i

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

# Convert chunks into numerical vectors
tfidf_matrix = vectorizer.fit_transform(chunks)

print("FDS Search Engine Created Successfully!")
print("Number of FDS chunks:", len(chunks))
print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Matrix shape:", tfidf_matrix.shape)

FDS Search Engine Created Successfully!
Number of FDS chunks: 24
Vocabulary size: 2432
Matrix shape: (24, 2432)


In [22]:
from sklearn.metrics.pairwise import cosine_similarity

def search_fds(question, top_k=3):

    # Convert question into TF-IDF vector
    question_vector = vectorizer.transform([question])

    # Compare question with all FDS chunks
    scores = cosine_similarity(
        question_vector,
        tfidf_matrix
    ).flatten()

    # Get the most relevant chunks
    top_indices = scores.argsort()[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "chunk_number": index + 1,
            "similarity_score": round(float(scores[index]), 3),
            "content": chunks[index]
        })

    return results

## 6. Gemini setup and Agentic AI
Set the Gemini API key in Colab Secrets as `GEMINI_API_KEY`. The agent selects relevant Python tools, receives their results and can make additional tool calls. No warehouse transactions are executed.

In [23]:
from google import genai
from google.genai import types
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(timeout=120000)
)

MODEL_NAME = "gemini-3.5-flash-lite"

print("Gemini client updated")

Gemini client updated


In [24]:
from google.colab import userdata
api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Add GEMINI_API_KEY to Google Colab Secrets before continuing.")

print("Gemini API key retrieved from Colab Secrets.")

Gemini API key retrieved from Colab Secrets.


### Persistent conversational memory
Google Drive stores conversation memory and historical observations. This notebook uses `/content/gdrive` consistently.

In [25]:
from google.colab import drive

drive.mount('/content/gdrive')

print("Google Drive connected successfully!")

Mounted at /content/gdrive
Google Drive connected successfully!


In [26]:
import os
import json

PROJECT_FOLDER = "/content/gdrive/MyDrive/Warehouse_AI_Project"
os.makedirs(PROJECT_FOLDER, exist_ok=True)

MEMORY_FILE = os.path.join(PROJECT_FOLDER, "conversation_memory.json")
OBSERVATION_FILE = os.path.join(PROJECT_FOLDER, "warehouse_observations.json")

for path in [MEMORY_FILE, OBSERVATION_FILE]:
    if not os.path.exists(path):
        with open(path, "w") as f:
            json.dump([], f, indent=4)

with open(MEMORY_FILE, "r") as f:
    conversation_memory = json.load(f)

print("Persistent memory loaded:", len(conversation_memory), "messages")
print("Project folder:", PROJECT_FOLDER)

Persistent memory loaded: 34 messages
Project folder: /content/gdrive/MyDrive/Warehouse_AI_Project


In [72]:
def save_to_memory(role, message):
    conversation_memory.append({"role": role, "message": message})
    with open(MEMORY_FILE, "w") as f:
        json.dump(conversation_memory, f, indent=4)


print("Conversational memory functions ready.")

Conversational memory functions ready.


In [28]:
MODEL_NAME = "gemini-3.5-flash-lite"

# Define Conversational Memory retrieval function
def retrieve_memory(question, top_k=3):

    if not conversation_memory:
        return []

    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    # Retrieve only user messages
    user_memory = [
        item for item in conversation_memory
        if item["role"] == "user"
    ]

    if not user_memory:
        return []

    memory_texts = [
        item["message"] for item in user_memory
    ]

    vectorizer = TfidfVectorizer(stop_words="english")

    memory_vectors = vectorizer.fit_transform(
        memory_texts + [question]
    )

    scores = cosine_similarity(
        memory_vectors[-1],
        memory_vectors[:-1]
    ).flatten()

    top_indices = scores.argsort()[::-1]

    results = []
    seen_messages = set()

    for index in top_indices:

        message = user_memory[index]["message"]
        normalized_message = message.strip().lower()

        if normalized_message in seen_messages:
            continue

        if scores[index] > 0:

            results.append({
                "role": "user",
                "message": message,
                "similarity": round(
                    float(scores[index]), 3
                )
            })

            seen_messages.add(normalized_message)

        if len(results) >= top_k:
            break

    return results

# Define Agent schemas
inventory_tool = {
    "type": "function",
    "name": "get_sku_status",
    "description": "Gets inventory, demand, and operational status for a specific SKU.",
    "parameters": {
        "type": "object",
        "properties": {
            "sku": {"type": "string", "description": "Warehouse SKU identifier"}
        },
        "required": ["sku"]
    }
}

stockout_tool = {
    "type": "function",
    "name": "get_stockout_risks",
    "description": "Gets all SKUs with high or medium stockout risk.",
    "parameters": {"type": "object", "properties": {}}
}

replenishment_tool = {
    "type": "function",
    "name": "get_replenishment_actions",
    "description": "Gets replenishment recommendations, existing task information, and reserve shortage exceptions.",
    "parameters": {"type": "object", "properties": {}}
}

fds_search_tool = {
    "type": "function",
    "name": "search_fds",
    "description": "Searches the Warehouse Functional Design Specification for relevant business rules and replenishment logic.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The warehouse functional question to search for."},
            "top_k": {"type": "integer", "description": "Number of relevant FDS chunks to retrieve."}
        },
        "required": ["question"]
    }
}

warehouse_tools = [inventory_tool, stockout_tool, replenishment_tool, fds_search_tool]

tool_functions = {
    "get_sku_status": get_sku_status,
    "get_stockout_risks": get_stockout_risks,
    "get_replenishment_actions": get_replenishment_actions,
    "search_fds": search_fds
}

agent_system_prompt = """
You are an Agentic AI Warehouse Operations Copilot
with WMS Functional Consultant expertise.

Use these tools when relevant:
1. get_sku_status
2. get_stockout_risks
3. get_replenishment_actions
4. search_fds

Investigate the user's warehouse question.
Check inventory, demand, open replenishment tasks,
reserve availability and relevant FDS rules.

Do not invent quantities or task statuses.
Identify missing information and discrepancies.
Never execute warehouse transactions.
Recommendations require human approval.

Provide:
A. Investigation Summary
B. Inventory and Demand
C. Stockout Risk
D. Replenishment Analysis
E. FDS Rule Comparison
F. Discrepancies
G. Consultant Recommendations
H. Missing Information

IMPORTANT DATA ACCURACY RULES:

1. Always prioritize current Python tool results
   over information retrieved from conversational memory.

2. Never use an older assistant-generated report
   as evidence of current inventory or task status.

3. Use replenishment quantities exactly as returned
   by get_replenishment_actions.

4. Do not treat reserve quantity already committed
   to an existing task as additional available stock.

5. Clearly distinguish physical inventory,
   committed reserve, eligible reserve,
   net required quantity and unfulfilled quantity.

6. If a calculation is not directly supported
   by tool results, state that it needs validation.

   IMPORTANT REPLENISHMENT REPORTING RULES:

1. Always use the latest Python tool outputs
   as the source of truth.

2. Net Required Qty must be explained using
   Projected_FP_With_Tasks, not Current_Forward_Pick.

3. Never use outdated reserve shortage values
   from previous conversations.

4. Do not calculate reserve shortage by subtracting
   Gross Reserve Qty from Net Required Qty when
   that reserve is already committed to an
   existing replenishment task.

5. Clearly distinguish:
   - Current Forward Pick Qty
   - Projected Forward Pick Qty
   - Existing Open Task Qty
   - Gross Reserve Qty
   - Reserve Committed Qty
   - Eligible Available Reserve Qty
   - Net Required Qty
   - Recommended Task Qty
   - Unfulfilled Qty

6. An existing replenishment task is not an
   inbound purchase order or ASN.

7. Do not interpret Unfulfilled_Qty = 0 as
   proof that the warehouse has no inventory
   shortage. Explain the task action and its
   limitations.

8. If a shortage calculation is not explicitly
   supported by the tool output, do not invent
   a shortage quantity. State that further
   validation is required.

   IMPORTANT PROJECTION AND REPLENISHMENT RULES:

1. Treat replenishment_recommendation_df as
   the authoritative source for final replenishment
   calculations.

2. Net_Required_Qty is calculated using:
   Target_Qty - Projected_FP_With_Tasks.

3. Projected_FP_With_Tasks includes:
   Current forward-pick inventory,
   forecast lead-time demand deduction,
   and existing open replenishment task quantity.

4. Do not use the simplified calculation:
   Current_Forward_Pick + Existing_Open_Task_Qty
   as a replacement for the canonical projection.

5. If get_sku_status reports a different projected
   quantity, explain that it uses a different projection
   method. Do not silently combine the values.

6. Unfulfilled_Qty = 0 must NOT be interpreted
   as proof that no inventory shortage exists.

7. If Eligible_Available_Reserve = 0 and
   Recommended_Task_Qty = 0, explain that no new
   replenishment task is recommended by the current
   engine because existing work is already present.

8. Clearly distinguish:
   Net_Required_Qty,
   Recommended_Task_Qty,
   Eligible_Available_Reserve,
   Unfulfilled_Qty.

9. Never claim that 82 units are available or
   that the requirement has been fully satisfied
   merely because an existing task exists.

   ADDITIONAL REPORT ACCURACY RULES:

1. Do not classify Current_Forward_Pick and
   Projected_FP_With_Tasks as a discrepancy.
   They represent different inventory measures.

2. Explain that projected inventory includes
   forecast lead-time demand and existing
   replenishment task quantities.

3. Unfulfilled_Qty = 0 does not mean the
   Net_Required_Qty has been fulfilled.

4. Do not claim that all reserve inventory
   has been successfully replenished.
   Distinguish committed reserve from
   physically available reserve.

5. If task execution status is unavailable,
   recommend verifying task status before
   recommending escalation or expediting.

6. Do not assume that an open task is
   already assigned or being executed.

7. In the Discrepancies section, report
   only genuine data or calculation conflicts.
   Do not report differences between
   current and projected inventory as errors.

8. Clearly state that recommendations
   are simulated and no real WMS transaction
   has been executed.
"""

In [29]:
import json

def run_warehouse_agent(user_question, max_rounds=4):

    print("WAREHOUSE AGENT STARTED")
    print("=" * 60)

    # STEP 1: Retrieve relevant previous conversations
    previous_context = retrieve_memory(user_question, top_k=3)

    memory_context = "\n".join([
        f"{item['role']}: {item['message']}"
        for item in previous_context
    ])

    if not memory_context:
        memory_context = "No relevant previous conversation found."

    print("\nRelevant memory retrieved:")
    print(memory_context)

    # STEP 2: Send memory and current question to Gemini
    response = client.interactions.create(
        model=MODEL_NAME,
        input=[
            {"type": "text", "text": agent_system_prompt},
            {
                "type": "text",
                "text": "PREVIOUS CONVERSATION:\n" + memory_context
            },
            {
                "type": "text",
                "text": "CURRENT QUESTION:\n" + user_question
            }
        ],
        tools=warehouse_tools
    )

    total_tool_calls = 0

    # STEP 3: Agentic tool orchestration
    for round_number in range(1, max_rounds + 1):

        function_steps = [
            step for step in response.steps
            if step.type == "function_call"
        ]

        if not function_steps:
            print("\nAgent completed its investigation.")
            break

        print(f"\nInvestigation Round: {round_number}")

        function_results = []

        for step in function_steps:

            tool_name = step.name
            arguments = step.arguments or {}

            if isinstance(arguments, str):
                arguments = json.loads(arguments)

            print("\nGemini selected:", tool_name)
            print("Arguments:", arguments)

            try:
                if tool_name not in tool_functions:
                    raise ValueError(f"Unknown tool: {tool_name}")

                result = tool_functions[tool_name](**arguments)

                print("Tool executed successfully.")

                result_text = json.dumps(result, default=str)

            except Exception as error:

                print("Tool execution error:", error)

                result_text = json.dumps({
                    "error": str(error),
                    "tool": tool_name
                })

            function_results.append({
                "type": "function_result",
                "name": tool_name,
                "call_id": step.id,
                "result": [
                    {"type": "text", "text": result_text}
                ]
            })

            total_tool_calls += 1

        print("\nSending tool results back to Gemini...")

        response = client.interactions.create(
            model=MODEL_NAME,
            previous_interaction_id=response.id,
            tools=warehouse_tools,
            input=function_results
        )

    else:
        print("\nMaximum investigation rounds reached.")

    # STEP 4: Save current conversation into memory
    final_answer = response.output_text

    save_to_memory("user", user_question)
    save_to_memory("assistant", final_answer)

    print("\nTotal Tool Calls:", total_tool_calls)
    print("Conversation saved to memory.")
    print("=" * 60)

    return final_answer

### Agent demonstration
Run this when you want to demonstrate multi-tool investigation. It calls Gemini and may take some time depending on API response.

In [43]:
agent_answer = run_warehouse_agent(
    "Investigate SKU1005. Check inventory, stockout risk, replenishment actions and relevant FDS rules. Identify discrepancies and recommend next steps. Do not execute warehouse transactions.",
    max_rounds=4
)
print(agent_answer)

WAREHOUSE AGENT STARTED

Relevant memory retrieved:
user: Investigate SKU1007. Check inventory, stockout risk, replenishment actions and relevant FDS rules. Identify discrepancies and recommend next steps. Do not execute warehouse transactions.
user: Investigate SKU1006. Check inventory, stockout risk, replenishment actions and relevant FDS rules. Identify discrepancies and recommend next steps. Do not execute warehouse transactions.
user: Investigate SKU1007 stockout risk.

Investigation Round: 1

Gemini selected: get_sku_status
Arguments: {'sku': 'SKU1005'}
Tool executed successfully.

Gemini selected: get_stockout_risks
Arguments: {}
Tool executed successfully.

Gemini selected: get_replenishment_actions
Arguments: {}
Tool executed successfully.

Gemini selected: search_fds
Arguments: {'question': 'replenishment trigger forward pick reserve shortage target quantity'}
Tool executed successfully.

Sending tool results back to Gemini...

Agent completed its investigation.

Total Tool C

## 7. Historical warehouse observations and pattern detection
The function below can capture a real notebook snapshot into Drive. Run the capture call only when you intentionally want to append a new observation set. The following demo snapshots are simulated for portfolio demonstration.

In [52]:
from datetime import datetime
import json
import os

# WAREHOUSE HISTORICAL OBSERVATION STORAGE

def save_warehouse_observations():

    observations = []
    snapshot_time = datetime.now().isoformat()

    # Capture the current warehouse snapshot
    for _, row in analytics_df.iterrows():

        observation = {
            "observed_at": snapshot_time,
            "SKU": str(row["SKU"]),
            "Velocity_Class": str(row["Velocity_Class"]),
            "Forward_Pick_Qty": float(row["Forward_Pick_Qty"]),
            "Reserve_Qty": float(row["Reserve_Qty"]),
            "Minimum_Qty": float(row["Minimum_Qty"]),
            "Maximum_Qty": float(row["Maximum_Qty"]),
            "Average_Daily_Demand": float(row["Average_Daily_Demand"]),
            "Stockout_Risk": str(row["Stockout_Risk"]),
            "Forward_Pick_Days_Cover": float(row["Forward_Pick_Days_Cover"])
        }

        observations.append(observation)

    # Load existing historical observations
    if os.path.exists(OBSERVATION_FILE):

        with open(OBSERVATION_FILE, "r") as file:
            historical_data = json.load(file)

    else:
        historical_data = []

    # Append current observations
    historical_data.extend(observations)

    # Save updated history to Google Drive
    with open(OBSERVATION_FILE, "w") as file:

        json.dump(
            historical_data,
            file,
            indent=4
        )

    print("Warehouse observations saved successfully!")
    print("New observations:", len(observations))
    print("Total historical observations:", len(historical_data))


print("Historical observation storage function READY")

Historical observation storage function READY


In [56]:
import pandas as pd
import json
import os

# Load historical warehouse observations

if not os.path.exists(OBSERVATION_FILE):

    historical_observations = []

    print("No historical observations file found.")
    print("Please capture a warehouse snapshot first.")

else:

    with open(OBSERVATION_FILE, "r") as file:
        historical_observations = json.load(file)

# Convert JSON into DataFrame

observations_df = pd.DataFrame(historical_observations)

print("HISTORICAL WAREHOUSE DATA")
print("=" * 50)

print("Total observations:", len(observations_df))

if not observations_df.empty:

    print("Unique SKUs:", observations_df["SKU"].nunique())

    display(observations_df.head(10))

else:

    print("No historical warehouse records available.")

HISTORICAL WAREHOUSE DATA
Total observations: 20
Unique SKUs: 10


,observed_at,SKU,Velocity_Class,Forward_Pick_Qty,Reserve_Qty,Minimum_Qty,Maximum_Qty,Average_Daily_Demand,Stockout_Risk,Forward_Pick_Days_Cover
0,2026-11-14T13:48:21.559743,SKU1001,Fast,25.0,200.0,30.0,100.0,35.67,MEDIUM RISK,0.70
1,2026-11-14T13:48:21.560128,SKU1002,Medium,80.0,150.0,40.0,120.0,21.77,LOW RISK,3.67
2,2026-11-14T13:48:21.560226,SKU1003,Fast,15.0,50.0,25.0,80.0,41.67,HIGH RISK,0.36
3,2026-11-14T13:48:21.560313,SKU1004,Slow,45.0,300.0,30.0,100.0,15.67,LOW RISK,2.87
4,2026-11-14T13:48:21.560387,SKU1005,Fast,20.0,0.0,20.0,90.0,31.77,HIGH RISK,0.63
5,2026-11-14T13:48:21.560454,SKU1006,Medium,55.0,120.0,35.0,110.0,26.97,LOW RISK,2.04
6,2026-11-14T13:48:21.560520,SKU1007,Fast,12.0,18.0,25.0,100.0,47.67,HIGH RISK,0.25
7,2026-11-14T13:48:21.560583,SKU1008,Slow,90.0,400.0,50.0,150.0,9.47,LOW RISK,9.50
8,2026-11-14T13:48:21.560654,SKU1009,Medium,32.0,70.0,35.0,100.0,29.67,MEDIUM RISK,1.08
9,2026-11-14T13:48:21.560718,SKU1010,Fast,40.0,25.0,30.0,120.0,45.67,HIGH RISK,0.88


In [62]:
def detect_stockout_patterns(observations_df):

    pattern_df = observations_df.copy()

    # Identify forward-pick minimum breaches
    pattern_df["Below_Minimum"] = (
        pattern_df["Forward_Pick_Qty"]
        < pattern_df["Minimum_Qty"]
    )

    # Group historical observations by SKU
    stockout_patterns = (
        pattern_df.groupby("SKU")
        .agg(
            Total_Observations=("SKU", "count"),
            High_Risk_Count=(
                "Stockout_Risk",
                lambda x: (x == "HIGH RISK").sum()
            ),
            Medium_Risk_Count=(
                "Stockout_Risk",
                lambda x: (x == "MEDIUM RISK").sum()
            ),
            Below_Minimum_Count=(
                "Below_Minimum", "sum"
            ),
            Average_FP_Qty=(
                "Forward_Pick_Qty", "mean"
            ),
            Average_Daily_Demand=(
                "Average_Daily_Demand", "mean"
            )
        )
        .reset_index()
    )

    # Calculate high-risk frequency
    stockout_patterns["High_Risk_Percentage"] = (
        stockout_patterns["High_Risk_Count"]
        / stockout_patterns["Total_Observations"]
        * 100
    ).round(2)

    # Classify historical patterns
    def classify_pattern(row):

        if row["Total_Observations"] < 2:
            return "INSUFFICIENT HISTORY"

        elif row["High_Risk_Count"] >= 2:
            return "RECURRING HIGH RISK"

        elif row["Below_Minimum_Count"] >= 2:
            return "REPEATED MINIMUM BREACH"

        else:
            return "NO REPEATED HIGH-RISK PATTERN"

    stockout_patterns["Historical_Pattern"] = (
        stockout_patterns.apply(
            classify_pattern, axis=1
        )
    )

    return stockout_patterns


# Execute historical pattern detection
actual_stockout_patterns_df = detect_stockout_patterns(
    observations_df
)

print("HISTORICAL STOCKOUT PATTERN ANALYSIS")
print("=" * 60)

display(actual_stockout_patterns_df)

HISTORICAL STOCKOUT PATTERN ANALYSIS


,SKU,Total_Observations,High_Risk_Count,Medium_Risk_Count,Below_Minimum_Count,Average_FP_Qty,Average_Daily_Demand,High_Risk_Percentage,Historical_Pattern
0,SKU1001,2,0,2,2,25.0,35.67,0.0,REPEATED MINIMUM BREACH
1,SKU1002,2,0,0,0,80.0,21.77,0.0,NO REPEATED HIGH-RISK PATTERN
2,SKU1003,2,2,0,2,15.0,41.67,100.0,RECURRING HIGH RISK
3,SKU1004,2,0,0,0,45.0,15.67,0.0,NO REPEATED HIGH-RISK PATTERN
4,SKU1005,2,2,0,0,20.0,31.77,100.0,RECURRING HIGH RISK
5,SKU1006,2,0,0,0,55.0,26.97,0.0,NO REPEATED HIGH-RISK PATTERN
6,SKU1007,2,2,0,2,12.0,47.67,100.0,RECURRING HIGH RISK
7,SKU1008,2,0,0,0,90.0,9.47,0.0,NO REPEATED HIGH-RISK PATTERN
8,SKU1009,2,0,2,2,32.0,29.67,0.0,REPEATED MINIMUM BREACH
9,SKU1010,2,2,0,0,40.0,45.67,100.0,RECURRING HIGH RISK


In [60]:
import pandas as pd

# Keep the original snapshot unchanged
current_snapshot = observations_df.copy()
current_snapshot["Data_Source"] = "CURRENT_TRAINING_SNAPSHOT"

simulated_snapshots = []

# Create two earlier demonstration snapshots
for snapshot_date in ["2026-11-12", "2026-11-13"]:

    for _, row in observations_df.iterrows():

        record = row.to_dict()
        sku = record["SKU"]

        record["observed_at"] = (
            snapshot_date + "T09:00:00"
        )
        record["Data_Source"] = "SIMULATED_DEMO"

        # Simulate recurring high-risk conditions
        if sku == "SKU1007":
            record["Forward_Pick_Qty"] = 18 if snapshot_date == "2026-11-12" else 15
            record["Stockout_Risk"] = "HIGH RISK"

        elif sku == "SKU1003":
            record["Forward_Pick_Qty"] = 20 if snapshot_date == "2026-11-12" else 18
            record["Stockout_Risk"] = "HIGH RISK"

        elif sku == "SKU1005":
            record["Reserve_Qty"] = 0
            record["Stockout_Risk"] = "HIGH RISK"

        simulated_snapshots.append(record)

# Combine simulated history with current snapshot
simulated_history_df = pd.DataFrame(simulated_snapshots)

demo_observations_df = pd.concat(
    [simulated_history_df, current_snapshot],
    ignore_index=True
)

print("DEMONSTRATION HISTORY CREATED")
print("=" * 50)
print("Total observations:", len(demo_observations_df))
print("Unique SKUs:", demo_observations_df["SKU"].nunique())
print("Simulated observations:",
      (demo_observations_df["Data_Source"] == "SIMULATED_DEMO").sum())
print("Current observations:",
      (demo_observations_df["Data_Source"] == "CURRENT_TRAINING_SNAPSHOT").sum())

DEMONSTRATION HISTORY CREATED
Total observations: 60
Unique SKUs: 10
Simulated observations: 40
Current observations: 20


In [63]:
# Correct the two simulated historical records for SKU1010
mask = (
    (demo_observations_df["SKU"] == "SKU1010") &
    (demo_observations_df["Data_Source"] == "SIMULATED_DEMO")
)

demo_observations_df.loc[mask, "Stockout_Risk"] = "LOW RISK"

# Re-run pattern detection
demo_stockout_patterns_df = detect_stockout_patterns(
    demo_observations_df
)

display(
    demo_stockout_patterns_df.sort_values(
        by="High_Risk_Count",
        ascending=False
    )
)

,SKU,Total_Observations,High_Risk_Count,Medium_Risk_Count,Below_Minimum_Count,Average_FP_Qty,Average_Daily_Demand,High_Risk_Percentage,Historical_Pattern
4,SKU1005,6,6,0,0,20.000000,31.77,100.00,RECURRING HIGH RISK
2,SKU1003,6,6,0,6,17.666667,41.67,100.00,RECURRING HIGH RISK
6,SKU1007,6,6,0,6,15.000000,47.67,100.00,RECURRING HIGH RISK
9,SKU1010,6,2,0,0,40.000000,45.67,33.33,RECURRING HIGH RISK
1,SKU1002,6,0,0,0,80.000000,21.77,0.00,NO REPEATED HIGH-RISK PATTERN
0,SKU1001,6,0,6,6,25.000000,35.67,0.00,REPEATED MINIMUM BREACH
5,SKU1006,6,0,0,0,55.000000,26.97,0.00,NO REPEATED HIGH-RISK PATTERN
3,SKU1004,6,0,0,0,45.000000,15.67,0.00,NO REPEATED HIGH-RISK PATTERN
7,SKU1008,6,0,0,0,90.000000,9.47,0.00,NO REPEATED HIGH-RISK PATTERN
8,SKU1009,6,0,6,6,32.000000,29.67,0.00,REPEATED MINIMUM BREACH


In [66]:
# Convert Python pattern results into JSON
pattern_records = actual_stockout_patterns_df.to_dict(
    orient="records"
)

pattern_data = json.dumps(
    pattern_records,
    indent=2,
    default=str
)

print("Historical pattern data prepared for Gemini!")
print("Total SKU records:", len(pattern_records))

Historical pattern data prepared for Gemini!
Total SKU records: 10


### Gemini historical-pattern analysis
The model summarises the supplied Python results. Treat root-cause statements as investigation hypotheses unless supported by additional operational evidence.

In [67]:
warehouse_analysis_prompt = f"""
You are an AI Warehouse Operations Analyst
supporting a WMS Functional Consultant.

Analyse the following historical stockout
pattern detection results.

DATA:
{pattern_data}

Provide your analysis in these sections:

1. Warehouse Risk Summary
2. SKUs with Recurring High Risk
3. SKUs with Repeated Minimum Breaches
4. Root Cause Indicators
5. Recommended Warehouse Operational Actions
6. Functional Consultant Observations

RULES:
- Use only the supplied Python results.
- Do not invent inventory quantities.
- Distinguish high stockout risk from minimum breaches.
- Treat these observations as simulated test data.
- Do not claim actual historical warehouse trends.
- Recommendations must require human approval.
- Do not execute warehouse transactions.
"""

response = client.interactions.create(
    model=MODEL_NAME,
    input=warehouse_analysis_prompt
)

print("AI WAREHOUSE PATTERN ANALYSIS")
print("=" * 60)
print(response.output_text)

AI WAREHOUSE PATTERN ANALYSIS
# AI Warehouse Operations Analysis
**To:** WMS Functional Consultant  
**Subject:** Stockout Pattern Detection Analysis (Simulated Test Dataset)  
**Classification:** Internal Operational Review (Requires Human Approval)

---

### 1. Warehouse Risk Summary
Based on the analysis of the 10 simulated test SKUs (each evaluated across 2 total observations):
* **Recurring High Risk SKUs (100% High-Risk Percentage):** 4 SKUs (`SKU1003`, `SKU1005`, `SKU1007`, `SKU1010`). These exhibit critical stockout exposure.
* **Repeated Minimum Breach SKUs:** 2 SKUs (`SKU1001`, `SKU1009`). These operate below minimum thresholds but have not triggered high-risk classifications in all observed instances.
* **Stable / No Repeated High-Risk Pattern:** 4 SKUs (`SKU1002`, `SKU1004`, `SKU1006`, `SKU1008`). These currently maintain healthy footprints relative to demand.

---

### 2. SKUs with Recurring High Risk
The following SKUs show a 100% high-risk pattern across observations and

## 8. Interactive dashboard
The dashboard reuses the already-calculated historical pattern results and Gemini report.

In [70]:
# WAREHOUSE AI OPERATIONS DASHBOARD

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Calculate dashboard KPIs
total_skus = actual_stockout_patterns_df["SKU"].nunique()

recurring_high_risk = (
    actual_stockout_patterns_df["Historical_Pattern"]
    == "RECURRING HIGH RISK"
).sum()

repeated_breaches = (
    actual_stockout_patterns_df["Historical_Pattern"]
    == "REPEATED MINIMUM BREACH"
).sum()

total_observations = (
    actual_stockout_patterns_df["Total_Observations"].sum()
)

print("WAREHOUSE AI OPERATIONS DASHBOARD")
print("=" * 55)
print("Total SKUs Analysed:", total_skus)
print("Recurring High-Risk SKUs:", recurring_high_risk)
print("Repeated Minimum Breaches:", repeated_breaches)
print("Total Observations:", total_observations)

WAREHOUSE AI OPERATIONS DASHBOARD
Total SKUs Analysed: 10
Recurring High-Risk SKUs: 4
Repeated Minimum Breaches: 2
Total Observations: 20


In [71]:
# HISTORICAL STOCKOUT RISK DISTRIBUTION

risk_distribution = (
    actual_stockout_patterns_df["Historical_Pattern"]
    .value_counts()
    .reset_index()
)

risk_distribution.columns = [
    "Risk_Category",
    "SKU_Count"
]

fig = px.bar(
    risk_distribution,
    x="Risk_Category",
    y="SKU_Count",
    title="Historical Stockout Risk Distribution",
    text="SKU_Count",
    labels={
        "Risk_Category": "Historical Risk Category",
        "SKU_Count": "Number of SKUs"
    }
)

fig.update_layout(
    xaxis_title="Historical Risk Category",
    yaxis_title="Number of SKUs",
    template="plotly_white"
)

fig.show()

In [40]:
# HISTORICAL SKU PATTERN ANALYSIS

dashboard_table = actual_stockout_patterns_df[
    [
        "SKU",
        "Total_Observations",
        "High_Risk_Count",
        "Medium_Risk_Count",
        "Below_Minimum_Count",
        "High_Risk_Percentage",
        "Historical_Pattern"
    ]
].sort_values(
    by=["High_Risk_Count", "Below_Minimum_Count"],
    ascending=False
)

display(dashboard_table)

,SKU,Total_Observations,High_Risk_Count,Medium_Risk_Count,Below_Minimum_Count,High_Risk_Percentage,Historical_Pattern
2,SKU1003,3,3,0,3,100.00,RECURRING HIGH RISK
6,SKU1007,3,3,0,3,100.00,RECURRING HIGH RISK
4,SKU1005,3,3,0,0,100.00,RECURRING HIGH RISK
9,SKU1010,3,1,0,0,33.33,NO REPEATED HIGH-RISK PATTERN
0,SKU1001,3,0,3,3,0.00,REPEATED MINIMUM BREACH
8,SKU1009,3,0,3,3,0.00,REPEATED MINIMUM BREACH
1,SKU1002,3,0,0,0,0.00,NO REPEATED HIGH-RISK PATTERN
3,SKU1004,3,0,0,0,0.00,NO REPEATED HIGH-RISK PATTERN
5,SKU1006,3,0,0,0,0.00,NO REPEATED HIGH-RISK PATTERN
7,SKU1008,3,0,0,0,0.00,NO REPEATED HIGH-RISK PATTERN


In [41]:
# AI-GENERATED WAREHOUSE CONSULTANT REPORT

from IPython.display import display, Markdown

print("AI WAREHOUSE CONSULTANT INSIGHTS")
print("=" * 60)

display(Markdown(response.output_text))

AI WAREHOUSE CONSULTANT INSIGHTS


# AI Warehouse Operations Analysis
**To:** WMS Functional Consultant  
**From:** AI Warehouse Operations Analyst  
**Subject:** Analysis of Simulated Historical Stockout Pattern Detection Results  

---

### 1. Warehouse Risk Summary
Based on the analysis of the 10 simulated test SKUs across 3 total observations each:
* **Recurring High Risk (100% High Risk Percentage):** 3 SKUs (SKU1003, SKU1005, SKU1007)
* **Intermittent High Risk (33.33% High Risk Percentage):** 1 SKU (SKU1010)
* **Repeated Minimum Breaches (Medium Risk / Below Minimum):** 2 SKUs (SKU1001, SKU1009)
* **No Repeated High-Risk Pattern:** 4 SKUs (SKU1002, SKU1004, SKU1006, SKU1008)

The data indicates a clear split between stable inventory profiles and high-velocity items suffering from severe replenishment lag or misconfigured parameters.

---

### 2. SKUs with Recurring High Risk
These items exhibit a **100% High Risk Percentage** across all observations and require immediate operational focus:
* **SKU1003:** 
  * High Risk Count: 3 / 3
  * Below Minimum Count: 3 / 3
  * Average Forward Pick (FP) Qty: 17.67
  * Average Daily Demand: 41.67
* **SKU1005:** 
  * High Risk Count: 3 / 3
  * Below Minimum Count: 0 / 3 *(Note: Stock is maintained above the minimum threshold despite high-risk status)*
  * Average Forward Pick (FP) Qty: 20.00
  * Average Daily Demand: 31.77
* **SKU1007:** 
  * High Risk Count: 3 / 3
  * Below Minimum Count: 3 / 3
  * Average Forward Pick (FP) Qty: 15.00
  * Average Daily Demand: 47.67

*Note: SKU1010 experienced 1 High Risk count out of 3 observations (33.33% risk percentage) with an Average FP Qty of 40.0 and Average Daily Demand of 45.67, categorized under "NO REPEATED HIGH-RISK PATTERN" due to low frequency.*

---

### 3. SKUs with Repeated Minimum Breaches
These items show consistent inventory dipping below defined minimum thresholds (Medium Risk) without reaching the critical high-risk classification across all observations:
* **SKU1001:** 
  * Medium Risk Count: 3 / 3
  * Below Minimum Count: 3 / 3
  * Average FP Qty: 25.00
  * Average Daily Demand: 35.67
  * Historical Pattern: REPEATED MINIMUM BREACH
* **SKU1009:** 
  * Medium Risk Count: 3 / 3
  * Below Minimum Count: 3 / 3
  * Average FP Qty: 32.00
  * Average Daily Demand: 29.67
  * Historical Pattern: REPEATED MINIMUM BREACH

---

### 4. Root Cause Indicators
* **Demand-Supply Imbalance:** For recurring high-risk items like SKU1003 and SKU1007, the Average Daily Demand (41.67 and 47.67 respectively) heavily outweighs the Average Forward Pick (FP) Quantity (17.67 and 15.00). This indicates that forward pick capacities or replenishment batch sizes are insufficient to cover daily consumption rates.
* **Min/Max Parameter Misalignment:** SKUs experiencing repeated minimum breaches (SKU1001, SKU1009) maintain low forward pick quantities relative to demand, suggesting that replenishment triggers (reorder points) may be set too low or replenishment lead times are lagging behind consumption velocity.
* **Anomalous High-Risk Without Minimum Breach:** SKU1005 recorded 3 High Risk counts despite a "0" Below Minimum Count, pointing toward potential safety stock parameter triggers or lead-time risk rules flagging the SKU despite inventory technically resting above the baseline minimum.

---

### 5. Recommended Warehouse Operational Actions
*(Note: All recommendations require human review and formal approval before implementation. No automated system changes or warehouse transactions are executed.)*

1. **Parameter Review:** Request WMS operations personnel to review and evaluate reorder points and minimum stock thresholds for SKU1003, SKU1005, and SKU1007 to better align with their high daily demand rates.
2. **Replenishment Strategy Assessment:** Prompt inventory planners to analyze replenishment batch sizes and frequency for SKU1001 and SKU1009 to prevent repeated minimum breaches.
3. **Capacity Audit:** Recommend a physical or slotting review for high-velocity SKUs (such as SKU1007) to determine if forward pick locations can physically support higher maximum inventory quantities to buffer against daily demand spikes.

---

### 6. Functional Consultant Observations
* **Pattern Classification Integrity:** The detection logic successfully categorizes risk profiles, cleanly isolating SKUs with constant critical exposure ("RECURRING HIGH RISK") from those experiencing chronic baseline erosion ("REPEATED MINIMUM BREACH").
* **Configuration vs. Execution Check:** The disparity between Average FP Qty and Average Daily Demand across the high-risk cohort suggests that system replenishment rules (e.g., Min/Max logic) may need tuning within the WMS to reflect current operational throughputs.
* **Test Data Boundary:** These observations are based on simulated test data; therefore, operational changes should not be applied globally until real-world validation and human approval are secured.

## Interview demonstration checklist
- Explain the warehouse business problem before showing code.
- Show one SKU investigation with the AI agent.
- Explain that Python tools provide the operational data and Gemini coordinates the investigation.
- Show an FDS question to explain RAG.
- Show the historical pattern table and dashboard.
- State clearly that the dataset and historical snapshots are simulated.
- Explain that this is decision support; human approval is required and no live WMS transaction is executed.

**Do not run the notebook's snapshot-capture function repeatedly just to refresh the dashboard.** The demonstration history is constructed in memory and does not need to be appended to the persistent observation file.